In [0]:
# ==========================================
# CELL 1: Configuration and Authentication
# ==========================================

from pyspark.sql import functions as F

storage_account = "healthcarestoragerev01"
container = "input"

storage_key = dbutils.secrets.get(
    scope="healthcare-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Define Silver and Gold paths
silver_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/silver"
)

gold_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/gold"
)

print("Storage authentication configured")
print("Silver path:", silver_path)
print("Gold path:", gold_path)

Storage authentication configured
Silver path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver
Gold path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold


In [0]:
# ==========================================
# CELL 2: Check Silver Tables
# ==========================================

display(
    dbutils.fs.ls(silver_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/departments/,departments/,0,1788280214000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/encounters/,encounters/,0,1788280281000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/insurance_claim_data/,insurance_claim_data/,0,1788280289000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/patients/,patients/,0,1788280296000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/providers/,providers/,0,1788280303000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/transactions/,transactions/,0,1788280310000


In [0]:
# ==========================================
# CELL 3: Load Silver Departments and Transactions
# ==========================================

department_df = (
    spark.read
    .format("delta")
    .load(f"{silver_path}/departments")
)

transaction_df = (
    spark.read
    .format("delta")
    .load(f"{silver_path}/transactions")
)

print("Department records:", department_df.count())
print("Transaction records:", transaction_df.count())

print("\nDepartment Schema:")
department_df.printSchema()

print("\nTransaction Schema:")
transaction_df.printSchema()

Department records: 20
Transaction records: 10000

Department Schema:
root
 |-- DeptID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- _bronze_loaded_at: timestamp (nullable = true)
 |-- _silver_load_timestamp: timestamp (nullable = true)


Transaction Schema:
root
 |-- TransactionID: string (nullable = true)
 |-- EncounterID: string (nullable = true)
 |-- PatientID: string (nullable = true)
 |-- ProviderID: string (nullable = true)
 |-- DeptID: string (nullable = true)
 |-- VisitDate: date (nullable = true)
 |-- ServiceDate: date (nullable = true)
 |-- PaidDate: date (nullable = true)
 |-- VisitType: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- AmountType: string (nullable = true)
 |-- PaidAmount: double (nullable = true)
 |-- ClaimID: string (nullable = true)
 |-- PayorID: string (nullable = true)
 |-- ProcedureCode: integer (nullable = true)
 |-- ICDCode: string (nullable = true)
 |-- LineOfBusiness: string (nullable = true)
 |-- MedicaidID:

In [0]:
# ==========================================
# CELL 4: Department Revenue
# ==========================================

from pyspark.sql.functions import col, sum as spark_sum

d = department_df.alias("d")
t = transaction_df.alias("t")

gold_department_revenue_df = (
    t.join(
        d,
        col("t.DeptID") == col("d.DeptID"),
        "left"
    )
    .groupBy(
        col("d.DeptID"),
        col("d.Name")
    )
    .agg(
        spark_sum(col("t.PaidAmount")).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

print("Department revenue calculated successfully")

display(
    gold_department_revenue_df.limit(10)
)

Department revenue calculated successfully


DeptID,Name,TotalRevenue
DEPT018,Psychiatry,227924.69987106323
DEPT014,Pulmonology,226913.12937927246
DEPT019,Endocrinology,221267.40016555786
DEPT012,Pathology,214891.83964157104
DEPT013,Surgery,212005.06993865967
DEPT007,Dermatology,210542.32036018372
DEPT003,Neurology,208393.67013549805
DEPT001,Emergency,204406.95950126648
DEPT002,Cardiology,204154.14960861206
DEPT005,Pediatrics,202694.57003211975


In [0]:
# ==========================================
# CELL 5: Save Department Revenue to Gold
# ==========================================

department_revenue_path = (
    f"{gold_path}/department_revenue"
)

(
    gold_department_revenue_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(department_revenue_path)
)

print("Department Revenue successfully written to Gold")
print("Location:", department_revenue_path)

Department Revenue successfully written to Gold
Location: abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/department_revenue


In [0]:
# CELL 6: Verify Department Revenue

from pyspark.sql import functions as F

# Read Department Revenue from Gold
department_revenue_df = spark.read.format("delta").load(
    "abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/department_revenue"
)

print(
    "Department Revenue records:",
    department_revenue_df.count()
)

display(
    department_revenue_df.orderBy(
        F.desc("TotalRevenue")
    )
)

Department Revenue records: 20


DeptID,Name,TotalRevenue
DEPT018,Psychiatry,227924.69987106323
DEPT014,Pulmonology,226913.12937927246
DEPT019,Endocrinology,221267.40016555786
DEPT012,Pathology,214891.83964157104
DEPT013,Surgery,212005.06993865967
DEPT007,Dermatology,210542.32036018372
DEPT003,Neurology,208393.67013549805
DEPT001,Emergency,204406.95950126648
DEPT002,Cardiology,204154.14960861206
DEPT005,Pediatrics,202694.57003211975
